In [ ]:
# --- repo bootstrap ---
import sys
from pathlib import Path

repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent

if not (repo / "src").exists():
    !git clone https://github.com/thinkthoughts/ion-transport-waveform-pipeline.git
    %cd ion-transport-waveform-pipeline
    repo = Path.cwd()

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

print("Repo root:", repo)


# 06 — Tradeoff Analysis

Explore tradeoffs between transport duration, control effort, and excitation.

```text
parameters → waveform → motion → excitation metrics
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.ion_transport_waveform.config import TrapConfig, IonSpecies
from src.ion_transport_waveform.transport_path import minimum_jerk_path
from src.ion_transport_waveform.motion_sim import simulate_ion_motion
from src.ion_transport_waveform.excitation_metrics import residual_amplitude

cfg = TrapConfig()
ion = IonSpecies()

fig_dir = repo / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

data_dir = repo / "data" / "simulation_outputs"
data_dir.mkdir(parents=True, exist_ok=True)


## 1. Sweep transport duration


In [ ]:
x0 = -160e-6
x1 = 160e-6

durations = np.array([5, 10, 20, 40, 80, 120, 200]) * 1e-6

amps = []
peak_speeds = []
peak_accels = []

for T in durations:
    t = np.linspace(0, T, 1200)
    path = minimum_jerk_path(t, x0=x0, x1=x1, duration=T)

    x_traj, v_traj = simulate_ion_motion(t, path, cfg.omega_rad_s)

    dt = t[1] - t[0]
    v = np.gradient(path, dt)
    a = np.gradient(v, dt)

    amps.append(residual_amplitude(x_traj, path))
    peak_speeds.append(np.max(np.abs(v)))
    peak_accels.append(np.max(np.abs(a)))

amps = np.array(amps)
peak_speeds = np.array(peak_speeds)
peak_accels = np.array(peak_accels)


## 2. Excitation vs duration


In [ ]:
plt.figure(figsize=(7.5, 4.5))
plt.loglog(durations * 1e6, amps, marker="o")
plt.xlabel("transport duration (µs)")
plt.ylabel("residual amplitude (m)")
plt.title("Excitation vs transport duration")
plt.tight_layout()
plt.savefig(fig_dir / "06_excitation_vs_duration.png", dpi=180)
plt.show()


## 3. Excitation vs peak velocity


In [ ]:
plt.figure(figsize=(7.5, 4.5))
plt.loglog(peak_speeds, amps, marker="o")
plt.xlabel("peak velocity (m/s)")
plt.ylabel("residual amplitude (m)")
plt.title("Excitation vs peak velocity")
plt.tight_layout()
plt.savefig(fig_dir / "06_excitation_vs_velocity.png", dpi=180)
plt.show()


## 4. Excitation vs peak acceleration


In [ ]:
plt.figure(figsize=(7.5, 4.5))
plt.loglog(peak_accels, amps, marker="o")
plt.xlabel("peak acceleration (m/s²)")
plt.ylabel("residual amplitude (m)")
plt.title("Excitation vs peak acceleration")
plt.tight_layout()
plt.savefig(fig_dir / "06_excitation_vs_acceleration.png", dpi=180)
plt.show()


## 5. Pareto-style tradeoff (velocity vs excitation)


In [ ]:
plt.figure(figsize=(7.5, 4.5))
plt.scatter(peak_speeds, amps)

for i, T in enumerate(durations):
    plt.text(peak_speeds[i], amps[i], f"{T*1e6:.0f}µs")

plt.xscale("log")
plt.yscale("log")
plt.xlabel("peak velocity (m/s)")
plt.ylabel("residual amplitude (m)")
plt.title("Tradeoff: speed vs excitation")
plt.tight_layout()
plt.savefig(fig_dir / "06_tradeoff_velocity_vs_excitation.png", dpi=180)
plt.show()


## 6. Save tradeoff data


In [ ]:
np.savez(
    data_dir / "tradeoff_analysis_06.npz",
    durations=durations,
    excitation=amps,
    peak_speeds=peak_speeds,
    peak_accels=peak_accels
)

print("saved tradeoff data")


## Final

You now have a full transport pipeline with quantified tradeoffs:

```text
geometry → path → waveform → motion → excitation → tradeoffs
```